In [1]:
import optuna
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier

DATA_DIR = Path.cwd().parent / 'data' / 'processed'
RANDOM_STATE = 25

X_train = pd.read_parquet(DATA_DIR / 'X_train.parquet')
y_train = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']
y_int, _ = pd.factorize(y_train, sort=True)
y_int = pd.Series(y_int, index=y_train.index)

numeric_cols     = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'bool']).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 1.0),
    }
    pipe = Pipeline([
        ('pre', preprocessor),
        ('clf', XGBClassifier(tree_method='hist', n_jobs=-1,
                              random_state=RANDOM_STATE, verbosity=0, **params)),
    ])
    scores = cross_val_score(pipe, X_train, y_int, cv=cv, scoring='f1_macro', n_jobs=1)
    return scores.mean()

study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=20)

print(f"Best macro-F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-07-28 16:21:58,843] A new study created in memory with name: no-name-8dd3a310-db4b-4c44-a787-261e4c05ef84
[I 2026-07-28 16:24:34,987] Trial 0 finished with value: 0.4090963528419585 and parameters: {'n_estimators': 709, 'max_depth': 7, 'learning_rate': 0.025815403999659686, 'subsample': 0.5929556160450896, 'colsample_bytree': 0.7055500639625566, 'min_child_weight': 2, 'reg_alpha': 0.684968744374135, 'reg_lambda': 0.4376110596596504}. Best is trial 0 with value: 0.4090963528419585.
[I 2026-07-28 16:26:03,707] Trial 1 finished with value: 0.39754524386864865 and parameters: {'n_estimators': 489, 'max_depth': 5, 'learning_rate': 0.0392955136671792, 'subsample': 0.5565203503471982, 'colsample_bytree': 0.7235154231337446, 'min_child_weight': 6, 'reg_alpha': 0.16198510384493825, 'reg_lambda': 0.5207187880526836}. Best is trial 0 with value: 0.4090963528419585.
[I 2026-07-28 16:27:23,441] Trial 2 finished with value: 0.3975120932924364 and parameters: {'n_estimators': 328, 'max_depth'

Best macro-F1: 0.4644
Best params: {'n_estimators': 325, 'max_depth': 10, 'learning_rate': 0.2875310588143992, 'subsample': 0.5014920258907353, 'colsample_bytree': 0.8842686645188366, 'min_child_weight': 1, 'reg_alpha': 0.9874245356428328, 'reg_lambda': 0.025864929727211272}
